# Practical 10: Feature Scaling (StandardScaler vs MinMaxScaler) & Train-Test Split

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program to perform feature scaling and dataset partitioning. Apply StandardScaler (Z-score normalization) and MinMaxScaler to numerical features, compare their transformed distributions, and split the dataset into training (80%) and testing (20%) sets using train_test_split(..., test_size=0.2, random_state=42).

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
rg = np.random.default_rng(99)
n = 110
age = rg.integers(23, 61, n).astype(float)
salary = np.round(28500 + age * 970 + rg.normal(0, 7000, n), -2)
frame = pd.DataFrame({
    "Country": rg.choice(["Norway", "Chile", "Kenya"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rg.normal(0, 8000, n) > 56500, "Yes", "No"),
})
frame.loc[rg.choice(n, 5, replace=False), "Age"] = np.nan
frame.loc[rg.choice(n, 5, replace=False), "Salary"] = np.nan
print(frame.head(11))

num_cols = ["Age", "Salary"]
frame[num_cols] = frame[num_cols].fillna(frame[num_cols].mean())
print(frame[num_cols].isna().sum())

scaled_std = pd.DataFrame(StandardScaler().fit_transform(frame[num_cols]), columns=num_cols)
scaled_mm = pd.DataFrame(MinMaxScaler().fit_transform(frame[num_cols]), columns=num_cols)

stats = ["mean", "std", "min", "max"]
print("=== Original ===")
print(frame[num_cols].describe().loc[stats].round(3))
print("\n=== StandardScaler (mean ≈ 0, std ≈ 1) ===")
print(scaled_std.describe().loc[stats].round(3))
print("\n=== MinMaxScaler (range [0, 1]) ===")
print(scaled_mm.describe().loc[stats].round(3))

X = pd.get_dummies(frame.drop(columns=["Purchased"]), columns=["Country"], dtype=int)
y = frame["Purchased"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=7)
print(f"Total rows: {len(X)} | Train: {X_train.shape} | Test: {X_test.shape}")

zscaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = zscaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = zscaler.transform(X_test[num_cols])

print("Scaled TRAIN stats (mean ≈ 0, std ≈ 1):")
print(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("\nScaled TEST stats (close to, but not exactly, 0/1 – as expected):")
print(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))

,Country,Age,Salary,Purchased
0,Kenya,59.0,90500.0,Yes
1,Kenya,42.0,67200.0,Yes
2,Kenya,51.0,80400.0,Yes
3,Chile,44.0,67100.0,Yes
4,Kenya,29.0,53000.0,No
5,Kenya,42.0,55800.0,Yes
6,Chile,58.0,76500.0,Yes
7,Chile,59.0,81000.0,Yes
8,Chile,59.0,86500.0,Yes
9,Norway,46.0,83800.0,Yes


Age       0
Salary    0
dtype: int64
=== Original ===

,Age,Salary
mean,42.019,69045.714
std,10.033,12331.380
min,24.000,39200.000
max,59.000,96300.000



=== StandardScaler (mean ≈ 0, std ≈ 1) ===

,Age,Salary
mean,0.000,0.000
std,1.005,1.005
min,-1.804,-2.431
max,1.700,2.220


...

,Age,Salary
mean,0.515,0.523
std,0.287,0.216
min,0.000,0.000
max,1.000,1.000


Total rows: 110 | Train: (93, 5) | Test: (17, 5)
Scaled TRAIN stats (mean ≈ 0, std ≈ 1):

,Age,Salary
mean,0.000,-0.000
std,1.005,1.005



Scaled TEST stats (close to, but not exactly, 0/1 – as expected):

,Age,Salary
mean,-0.009,-0.021
std,0.820,0.924
